# 03 — Train the centralised baseline

Trains one model on **all the training data pooled on one machine**. This is the
reference every federated run is measured against, and it is `test01` in the
results tables.

Dataset: `dataset/multi_subtype_80mm` — three-class molecular subtype, all three
cohorts (I-SPY2 + I-SPY1 + Duke), 2,063 patients, 80 mm physical crop, 8 spread
slices per patient.

**Where this stands.** Twenty-one runs across five data configurations and
thirteen architectures put this task between 0.55 and 0.63 macro-AUC. The ceiling
does not move with preprocessing, architecture, normalisation, augmentation or
freezing. That is a result, not a failure — a systematic review of 106 studies
reaches the same conclusion about MRI and molecular subtype — and it still carries
enough signal to measure what federation costs, which is what the thesis asks.

**Two numbers to keep in mind.** The trivial baseline on the test set is **0.5112**
(always predict the majority class), and the run-to-run noise floor is **0.067**
macro-AUC. A difference smaller than that is not a result.


## 1. Setup and hardware

This notebook runs unchanged on an NVIDIA machine and on a MacBook. The cell below
reports which device was selected and every adaptation that follows from it.


In [ ]:
# Reload edited project modules without restarting the kernel.
#
# Python caches imported modules in sys.modules and never re-reads the file, so
# editing anything under src/ leaves this kernel running the old version - which
# surfaces as an ImportError for a function you can see in the file. These two
# lines must run BEFORE the imports below to cover them.
#
# It is not infallible: it does not update already-instantiated objects whose
# class changed shape. After a long editing session, restart the kernel.
%load_ext autoreload
%autoreload 2

# Imports in timed stages. If this cell ever feels slow, the printout says which
# stage cost the time rather than leaving you guessing at a frozen cell.
import os, time, platform, sys
from pathlib import Path

# MUST be set BEFORE torch is imported, or it has no effect. It lets an operation
# Apple has not implemented on MPS fall back to the CPU instead of raising.
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

_t0 = time.perf_counter()
def _stage(label):
    global _t
    now = time.perf_counter()
    print(f"  {label:<34}{now - _t:5.2f} s", flush=True)
    _t = now
_t = _t0

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
_stage("numpy + pandas")

import torch
_stage("torch")

import matplotlib.pyplot as plt
from PIL import Image
_stage("matplotlib + pillow")

import dataset_config as config
from dataset_config import Config, TASKS, lr_for_model
from core.models import SUPPORTED
_stage("project config + models")

from core.training import run, get_device, describe_device
from core.data import (AugmentConfig, PROFILES, augment_for, apply_augment,
                       effective_num_workers, IMAGENET_MEAN, IMAGENET_STD)
_stage("core.training + core.data (sklearn)")

print(f"  {'TOTAL':<34}{time.perf_counter() - _t0:5.2f} s\n")

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

# ============================================================================
# HARDWARE - NVIDIA CUDA, then Apple GPU, then CPU
# ============================================================================
DEVICE = get_device()
IS_CUDA = DEVICE.type == "cuda"
IS_MPS  = DEVICE.type == "mps"

print(f"platform      : {platform.system()} {platform.machine()}")
print(f"torch         : {torch.__version__}")
print(f"device chosen : {describe_device(DEVICE)}")
print()
print("what is available on this machine:")
print(f"  CUDA (NVIDIA)     : {torch.cuda.is_available()}")
print(f"  MPS (Apple GPU)   : built={torch.backends.mps.is_built()}"
      f" available={torch.backends.mps.is_available()}")
print(f"  CPU               : always")
print()
print("adaptations that follow from the device:")
print(f"  mixed precision : {'on  (GradScaler is a CUDA construct)' if IS_CUDA else 'off (only CUDA has a working GradScaler)'}")
print(f"  pin_memory      : {'on' if IS_CUDA else 'off'}")
print(f"  dataloader jobs : {effective_num_workers(8)}  (forced to 0 on macOS)")
if IS_MPS:
    print(f"  MPS fallback    : PYTORCH_ENABLE_MPS_FALLBACK={os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK')}")
    print(f"  per-step sync   : on (torch.mps.synchronize)")
    print()
    print("  The Apple GPU trains correctly here. It was banned for a long time")
    print("  after producing a NaN loss, which turned out to be an asynchronous")
    print("  host-to-device copy from unpinned memory, not a fault in MPS.")
    print("  Measured over a full epoch: loss 1.1546 and train_acc 0.4366 here,")
    print("  against 1.1502 and 0.4237 on CPU, in 231 s against 589 s.")

# Force the CPU if you ever need to rule the GPU out:
#   os.environ["BREAST_FORCE_CPU"] = "1"   (before get_device)


## 2. The data

A look at what the network will actually see before anything is trained.


In [ ]:
CFG_PREVIEW = Config(pipeline="thesis", task="subtype",
                     cohorts=("spy2", "spy1", "duke"),
                     dataset_name="multi_subtype_80mm")
DATASET = CFG_PREVIEW.dataset_dir

assert DATASET.is_dir(), (
    f"{DATASET} does not exist. Build it first with 02_build_dataset.ipynb.")

meta = pd.read_csv(DATASET / "metadata.csv", low_memory=False)
pat = meta.drop_duplicates("pid")

print(f"{len(meta):,} images from {pat.pid.nunique():,} patients")
print(f"{len(meta) / pat.pid.nunique():.2f} slices per patient\n")
print(pd.crosstab(pat.split, pat.label_name, margins=True).to_string())

test_counts = pat[pat.split == "test"].label_name.value_counts()
print(f"\ntrivial baseline on test: {test_counts.max()}/{test_counts.sum()}"
      f" = {test_counts.max() / test_counts.sum():.4f}")
print("\npatients per cohort:")
print(pat.cohort.value_counts().to_string())


In [ ]:
# One real training image per cohort and class. R = pre-contrast,
# G = early post-contrast, B = late post-contrast, so the COLOUR of a voxel
# encodes how it took up and released the contrast agent.
from PIL import Image

COHORTS = ["spy2", "duke", "spy1"]
CLASSES = ["HRposHER2neg", "TripleNeg", "HER2pos"]
rng = np.random.default_rng(42)

fig, axes = plt.subplots(3, 3, figsize=(6.5, 7), layout="constrained")
for r, co in enumerate(COHORTS):
    for c, cl in enumerate(CLASSES):
        a = axes[r][c]; a.set_xticks([]); a.set_yticks([]); a.grid(False)
        sub = meta[(meta.cohort == co) & (meta.label_name == cl)]
        if sub.empty:
            a.text(0.5, 0.5, "none", ha="center", va="center"); continue
        pid = rng.choice(sub.pid.unique())
        rows = sub[sub.pid == pid].sort_values("slice_order")
        row = rows.iloc[len(rows) // 2]
        a.imshow(Image.open(DATASET / "images" / row.filename))
        a.set_title(f"{co} - {cl}", fontsize=8)
fig.suptitle("What the network sees (224x224 RGB, 0.357 mm/px)", fontsize=10)
plt.show()


## 3. Choose the model

Every architecture below is wired and was tested. ResNet-18 is the measured
winner — not because it scored highest, but because across 1.5M to 87.6M
parameters everything lands in the same 0.55-0.63 band, and it is the cheapest
adequate backbone. A linear probe with 4,098 trainable parameters reached 0.6813,
which is the strongest evidence that the ceiling is signal rather than capacity.


In [ ]:
for name, note in SUPPORTED.items():
    print(f"{name!r:<22} # {note}")


## 4. Training configuration

Every option, with the measurement behind it where one exists. The defaults are
the configuration the reported centralised baseline used.


In [ ]:
# ============================================================================
# MODEL — pick one. Every architecture below is wired and was tested.
# ============================================================================
MODEL = "resnet18"           # 11.2M  the measured winner on this task
# MODEL = "resnet34"         # 21.3M
# MODEL = "resnet50"         # 23.5M  the reference in the comparable literature
# MODEL = "efficientnet_b0"  #  4.0M
# MODEL = "efficientnet_b3"  # 10.7M
# MODEL = "convnext_tiny"    # 27.8M
# MODEL = "convnext_small"   # 49.5M
# MODEL = "mobilenet_v3_large"  # 4.2M
# MODEL = "densenet121"      #  7.0M
# MODEL = "vit_b_16"         # 85.8M  torchvision ViT
# MODEL = "vit_mae_base"     # 85.8M  facebook/vit-mae-base, the AUTHORS' model
# MODEL = "swin_t"           # 27.5M
# MODEL = "thda_resnet34"    # 21.3M  dual-attention ResNet, arXiv:2510.13897
#
# Architecture does not decide this task: from 1.5M to 87.6M parameters every one
# lands in the same 0.55-0.63 macro-AUC band, and a linear probe with 4,098
# trainable parameters reached 0.6813. Pick the cheapest adequate backbone.

SEED = 42                    # any integer. Run at least two - one seed is not a
                             # result, and the noise floor here is 0.067 macro-AUC

CFG = Config(
    # ---- what to train on ---------------------------------------------------
    pipeline="thesis",
    # pipeline="reference",              # the dataset authors' preprocessing rules

    task="subtype",                      # 3 classes: HR+/HER2-, TripleNeg, HER2+
    # task="her2",                       # binary: HER2+ vs rest
    # task="hr",                         # binary: HR+ vs rest
    # task="pcr",                        # binary: pathological complete response

    cohorts=("spy2", "spy1", "duke"),    # all three, as the experiments used
    # cohorts=("spy2",),                 # I-SPY2 only - halves the data but
    #                                    #   removes the cohort shortcut entirely
    # cohorts=("duke",),                 # Duke only

    dataset_name="multi_subtype_80mm",   # pinned: the folder every experiment
                                         # expects. Leave it unless you built a
                                         # different dataset in notebook 02
    model=MODEL,
    seed=SEED,

    # ---- optimisation ---------------------------------------------------------
    optimizer="adamw",                   # "adamw" | "sgd" (sgd uses momentum 0.9)

    learning_rate=lr_for_model(MODEL),   # per-model default; resnet18 -> 1e-4
    # learning_rate=3e-5,                # measured 0.6473
    # learning_rate=1e-4,                # measured BEST
    # learning_rate=3e-4,                # measured WORST, 0.5728

    weight_decay=5e-4,

    scheduler="cosine",                  # "cosine" | "plateau" | "none"
                                         # plateau needs history; cosine is what
                                         # the federated clients evaluate in closed
                                         # form, so use cosine to stay comparable

    batch_size=24,                       # 24 measured best
    # batch_size=8,                      # measured 0.6264
    # batch_size=64,                     # measured 0.6238

    epochs=10,
    early_stopping_patience=30,          # 0 disables early stopping entirely
                                         # (the federated campaign used 0)

    # ---- regularisation ---------------------------------------------------------
    dropout=0.5,                         # 0.0 disables it
    label_smoothing=0.1,                 # 0.0 disables it
    class_weighted_loss=True,            # False = unweighted. Weights are inverse
                                         # frequency counted per PATIENT, not slice
    mixup_alpha=0.0,                     # 0.0 = off. Never conclusive here

    # ---- transfer learning ----------------------------------------------------------
    freeze_until="layer3",               # freezes conv1+bn1+layer1+layer2 = 6.1% of
                                         # the parameters. This is what the reported
                                         # centralised baseline and every federated
                                         # client used, so keep it to stay comparable.
                                         # It cut the seed spread from 0.026 to 0.003
    # freeze_until="none",               # train the whole network
    # freeze_until="layer4",             # 25% of parameters. Never run - the honest
                                         #   freezing test, still open
                                         # also valid: "conv1" | "bn1" | "layer1" | "layer2"
    freeze_bn=False,                     # True also freezes BatchNorm statistics

    # ---- batching -----------------------------------------------------------------
    max_slices_per_patient_per_batch=1,  # 0 = no limit. Neighbouring slices of one
                                         # tumour are near-duplicates and give
                                         # almost the same gradient
    image_size=224,                      # the backbones' native size; changing it
                                         # forces interpolation inside the network

    # ---- evaluation ------------------------------------------------------------------
    aggregation="mean",                  # "mean" | "median" | "max"
                                         # how slice probabilities become one
                                         # prediction per patient. The authors
                                         # report median as best for HER2
    monitor_metric="auc",                # "auc" | "accuracy" | "balanced_accuracy"
                                         #       | "macro_f1"
                                         # what the best checkpoint is selected on.
                                         # NEVER a training metric

    # ---- hardware --------------------------------------------------------------------
    mixed_precision=True,                # honoured on CUDA, ignored on CPU
    num_workers=8,                       # forced to 0 on macOS by the loader

    augmentation="default",              # "default" | "half" | "none" | "custom"
                                         # "custom" is built in the next section
    notes="centralised baseline",        # free text, written into results.json
)

# MEASURED ON THIS MACHINE: one epoch over all 12,131 slices takes 589 s on the
# MacBook CPU at batch 8, so 100 epochs is roughly 16 hours. The reported baseline
# was trained on an NVIDIA RTX 4000 Ada in minutes.
#
# The loop itself is correct on CPU - the smoke test gave a finite loss of 1.1502
# and train_acc 0.4237, which is the expected shape. Uncomment to reproduce that
# check in about ten minutes before committing to a real run.
# if not IS_CUDA:
#     CFG.epochs = 1
#     CFG.batch_size = 8
#     CFG.early_stopping_patience = 0

print(f"model {CFG.model} · lr {CFG.learning_rate} · batch {CFG.batch_size}"
      f" · {CFG.epochs} epochs · seed {CFG.seed}")


## 5. Data augmentation — every option

Applied at load time, every epoch, to the **training split only**. The files on
disk are never augmented.

Three ready-made profiles exist — `default`, `half`, `none` — and the cell below
additionally builds a fully explicit one so every knob is visible and editable.

**This is the only regulariser in the project with a measured effect.** Halving it
(`half`) raised training accuracy from 0.57 to 0.99 and tripled the train/test gap
from 0.135 to 0.512. The current setting is what stops the model memorising.


In [ ]:
CUSTOM_AUGMENT = AugmentConfig(
    enabled=True,

    # ---- flips -----------------------------------------------------------
    horizontal_flip=0.5,        # reads as the contralateral breast
    vertical_flip=0.0,          # OFF: a cranio-caudal flip makes anatomy that
                                #      does not exist. Leave at 0.

    # ---- geometry: drawn independently, composed into ONE affine transform,
    #      so the image is interpolated once instead of three times ---------
    rotation_degrees=15.0,
    rotation_probability=1.0,
    scale_range=(0.9, 1.1),     # zoom
    scale_probability=1.0,
    translate_fraction=0.08,    # shift, as a fraction of the image
    translate_probability=1.0,

    # ---- intensity ---------------------------------------------------------
    brightness_range=(0.8, 1.2),
    brightness_probability=1.0,
    gaussian_noise_std=(0.005, 0.03),
    noise_probability=0.25,

    # ---- occlusion ----------------------------------------------------------
    cutout_probability=0.0,     # OFF by default; side is 10-30% of the image
)

# Registering it as a profile is what makes it selectable by name, which is how
# the trainer and the federated clients both ask for an augmentation set.
PROFILES["custom"] = CUSTOM_AUGMENT

# CFG.augmentation = "custom"   # uncomment to use the explicit set above
# CFG.augmentation = "half"     # measured: train acc 0.57 -> 0.99, gap tripled
# CFG.augmentation = "none"     # no augmentation at all

aug = augment_for(CFG.augmentation)
print(f"active profile: {CFG.augmentation!r}\n")
for f in AugmentConfig.__dataclass_fields__:
    print(f"  {f:<26} {getattr(aug, f)}")

print("\nAlways applied afterwards, to every split including validation and test:")
print(f"  ImageNet normalisation  mean={IMAGENET_MEAN.flatten().tolist()}")
print(f"                          std ={IMAGENET_STD.flatten().tolist()}")
print("  It runs LAST, so brightness and noise operate in [0,1] space where")
print("  clamping is meaningful.")


In [ ]:
# See the augmentation. Same image, six independent draws - this is what varies
# between epochs for a single training slice.
import random

row = meta[meta.split == "train"].sample(1, random_state=7).iloc[0]
png = Image.open(DATASET / "images" / row.filename)
base = torch.from_numpy(np.asarray(png).astype(np.float32) / 255.0).permute(2, 0, 1)

random.seed(3); torch.manual_seed(3)
fig, axes = plt.subplots(1, 7, figsize=(13, 2.4), layout="constrained")
axes[0].imshow(png); axes[0].set_title("on disk", fontsize=8)
for i in range(1, 7):
    out = apply_augment(base.clone(), aug)
    axes[i].imshow(np.clip(out.permute(1, 2, 0).numpy(), 0, 1))
    axes[i].set_title(f"draw {i}", fontsize=8)
for a in axes:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
fig.suptitle(f"Augmentation profile {CFG.augmentation!r} - a new draw every epoch",
             fontsize=10)
plt.show()


## 6. Confirm before spending time


In [ ]:
print(CFG.summary())
print()
print(f"device        : {DEVICE}")
print(f"augmentation  : {CFG.augmentation}")
print(f"output folder : {config.RESULTS_DIR}")
if not IS_CUDA:
    print()
    print("  Running on CPU. Measured on this machine: 589 s per epoch over all")
    print(f"  12,131 slices, so {CFG.epochs} epochs is about {CFG.epochs * 589 / 3600:.1f} hours.")
    print("  The loop is correct on CPU - only slow. Set CFG.epochs = 1 first.")


## 7. Train

One call. Everything is written into an auto-numbered folder under `results/`:
curves, ROC, precision-recall, confusion matrix, per-class metrics, the
classification report, per-patient predictions and both checkpoints.


In [ ]:
# progress=True shows a live batch counter for each epoch. It advances a bar and
# nothing else: printing a running loss would mean reading .item() every step,
# which is the per-batch synchronisation the loop removed for a measured 20% of
# step time on CUDA. The loss and accuracy arrive when the epoch ends.
run_dir = run(CFG, progress=True)

print("\nfiles written:")
for f in sorted(run_dir.rglob("*")):
    if f.is_file():
        print("  ", f.relative_to(run_dir))


## 8. Results


In [ ]:
import json

res = json.loads((run_dir / "results.json").read_text())
test = res.get("test", res)

print(f"macro-AUC          {test.get('auc', float('nan')):.4f}")
print(f"balanced accuracy  {test.get('balanced_accuracy', float('nan')):.4f}")
print(f"accuracy           {test.get('accuracy', float('nan')):.4f}")
print(f"trivial baseline   {test.get('trivial_baseline_accuracy', 0.5112):.4f}")
print()
print("Read any comparison against the 0.067 macro-AUC noise floor: a gap smaller")
print("than that is 'no difference detected', which is a finding. One seed is not")
print("a result - run seeds 1 and 42 before concluding anything.")

hist = run_dir / "history.csv"
if hist.is_file():
    h = pd.read_csv(hist)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot(h.index, h.train_loss, label="train loss")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
    col = "val_auc" if "val_auc" in h.columns else h.columns[-1]
    ax[1].plot(h.index, h[col], color="#D55E00", label=col)
    ax[1].set_xlabel("epoch"); ax[1].legend()
    fig.suptitle("Training curves", fontsize=10)
    plt.tight_layout(); plt.show()
